# Fast codec — optimization playground

ROS-ready inference for the v2 codec. **Toggle the optimizations** in the config cell and see the effect on
latency (including a growing message), plus a reconstruction to confirm quality is preserved.

- **fp16** — half precision (bit-identical PSNR)
- **compile** — `torch.compile`, fuses kernels (~2–3× compute)
- **dynamic** — compile once for dynamic H/W, so **changing / growing sizes don't recompile**
- **pin** — reused pinned host buffer for a fast CPU→GPU copy (messages come from CPU)
- **channels_last** — alternate memory format (usually a wash)

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
import fast_codec, data_multimodal as D

# ---- toggle optimizations here, then re-run the cells ----
FP16, COMPILE, DYNAMIC, PIN, CHANNELS_LAST = True, True, True, True, False

codec = fast_codec.FastCodec(fp16=FP16, compile=COMPILE, dynamic=DYNAMIC, pin=PIN, channels_last=CHANNELS_LAST)
print('opts:', codec.opts)

## Growing-message latency
`1st call` includes any compile; with **dynamic** on, new sizes after the first are stall-free.
`compute` = pure codec on GPU; `end-to-end` adds the CPU↔GPU copy of the full grid.

In [ ]:
SIZES = [(512,512),(1024,1024),(1536,1536),(2048,2048),(3000,3000),(4000,4000)]  # a message growing in size
fast_codec.benchmark(codec, SIZES, depth=2)   # depth 2 = 16x; try depth=3 (64x)

## Reconstruct a real grid — confirm quality is preserved

In [ ]:
MODALITY, H, W, DEPTH = 'rgb', 512, 768, 2      # any modality / size / rate
FILES = D.file_lists()
g = D._resize(D._load_native(MODALITY, FILES, 3), H, W).numpy()   # (nch,H,W) on CPU, like a ROS message
import time; _ = codec.roundtrip(g, DEPTH)                        # warm
t = time.perf_counter(); r = codec.roundtrip(g, DEPTH); torch.cuda.synchronize(); ms = (time.perf_counter()-t)*1000
psnr = 10*np.log10(1/max(((r-g)**2).mean(),1e-9))
d = lambda im: im.transpose(1,2,0) if im.shape[0]==3 else im[0]
fig, ax = plt.subplots(1,2,figsize=(13,5))
ax[0].imshow(d(g), cmap=None if g.shape[0]==3 else 'gray'); ax[0].set_title('original'); ax[0].axis('off')
ax[1].imshow(d(r), cmap=None if r.shape[0]==3 else 'gray'); ax[1].set_title(f'reconstructed — PSNR {psnr:.1f} dB, roundtrip {ms:.2f} ms'); ax[1].axis('off')
plt.tight_layout(); plt.show()

## Compare configurations (which optimizations help most)

In [ ]:
import time
configs = {
    'fp32 eager':        dict(fp16=False, compile=False, pin=False),
    'fp16':              dict(fp16=True,  compile=False, pin=True),
    'fp16+compile':      dict(fp16=True,  compile=True,  dynamic=True, pin=True),
}
H = W = 2048; g = np.random.rand(1,H,W).astype(np.float32)
print(f'compute-only enc+dec @ {W}x{H} (16x):')
for name, opt in configs.items():
    c = fast_codec.FastCodec(**opt)
    x = c._to_gpu(g)
    for _ in range(6): c.compute_only(x, 2, H, W)
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(15): c.compute_only(x, 2, H, W)
    torch.cuda.synchronize(); print(f'  {name:16s} {(time.perf_counter()-t)/15*1000:6.2f} ms')
    del c; torch.cuda.empty_cache()